In [ ]:
import chipwhisperer as cw
import time
import numpy as np
from IPython.display import clear_output
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import struct
from utils import *
import bz2
import pandas as pd
import sys
import optuna
from optuna.trial import TrialState

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
import numpy as np
import pandas as pd
import ast
import os
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

def parse_predictions(val):
    # Already a Python list
    if isinstance(val, list):
        return val
    # Missing / empty -> no prediction
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []
    # If it's a plain int (or numpy int), wrap it
    if isinstance(val, (int, np.integer)):
        return [int(val)]
    # Strings like "7", "[]", "[1,2]" -> eval safely
    if isinstance(val, str):
        s = val.strip()
        if s == "" or s == "[]":
            return []
        try:
            obj = ast.literal_eval(s)
        except Exception:
            # Fallback: maybe it's "7"
            try:
                return [int(s)]
            except Exception:
                return []
        # Normalize to list
        if isinstance(obj, list):
            return obj
        if isinstance(obj, (int, np.integer)):
            return [int(obj)]
        if isinstance(obj, tuple):
            return list(obj)
        return []
    # Anything else -> no prediction
    return []

def print_confusion_matrix(data_frame):
    # Define the labels for actual and predicted classes
    actual_labels = list(range(10))
    predicted_labels = list(range(10)) + ["Others", "Reset"]

    # Initialize an empty confusion matrix with zeros
    CM_df = pd.DataFrame(0, index=actual_labels, columns=predicted_labels)

    # Populate the confusion matrix by iterating over the data
    for _, row in data_frame.iterrows():
        actual = int(row["Actual"])
        predictions = parse_predictions(row["Prediction"])

        # If predictions list is empty, count as "Reset"
        if not predictions:
            CM_df.at[actual, "Reset"] += 1
        else:
            # Check if each predicted class is within the defined range, else mark as "Others"
            for pred in predictions:
                if pred in CM_df.columns:
                    CM_df.at[actual, pred] += 1
                else:
                    CM_df.at[actual, "Others"] += 1
    return CM_df

def normalize_confusion_matrix(CM_df):
    """
    Normalize the confusion matrix rows (classes 0–9 only) to percentages.
    Does NOT modify the input CM_df.
    """
    # Use only digit columns (0–9)
    digit_cols = [c for c in CM_df.columns if isinstance(c, int)]
    CM_core = CM_df[digit_cols].copy()

    CM_normalized = CM_core.div(CM_core.sum(axis=1), axis=0) * 100  # Convert to percentages
    CM_normalized = CM_normalized.fillna(0)  # Replace NaN with 0 for rows with no samples
    return CM_normalized

def visualize_confusion_matrix(CM_df):
    plt.figure(figsize=(10, 8))  # Adjust the figure size as needed
    sns.heatmap(CM_df, annot=True, fmt="d", cmap="YlGnBu", cbar=True)
    plt.title("Confusion Matrix")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.show()

def compute_reset_rate(confusion_matrix):
    """
    Compute reset rate as: total 'Reset' count / total samples.
    Here total samples = sum of all entries in the confusion matrix.
    """
    if "Reset" not in confusion_matrix.columns:
        reset_rate = 0.0
    else:
        reset_count = confusion_matrix["Reset"].sum()
        total_count = confusion_matrix.values.sum()
        reset_rate = reset_count / total_count if total_count > 0 else 0.0

    print("Reset rate: %.2f%%" % (reset_rate * 100))
    return reset_rate

def visualize_confusion_matrix_with_accuracy(CM_df, accuracy, reset_rate, save_path, title="Confusion Matrix"):
    """
    Visualize the normalized confusion matrix with accuracy and reset rate in the title.
    """
    plt.figure(figsize=(5, 4))
    sns.heatmap(CM_df, annot=True, fmt=".2f", cmap="YlGnBu", cbar=False, annot_kws={"size": 10})
    plt.title(f"{title}\nAccuracy: {accuracy:.2%} | Reset rate: {reset_rate:.2%}", fontsize=14)
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.tight_layout()
    # plt.savefig(save_path)
    plt.show()

def print_accuracy(confusion_matrix):
    # Calculate true positives (diagonal elements) for classes 0–9
    true_positives = sum(confusion_matrix.iloc[i, i] for i in range(10))

    # Exclude "Others" and "Reset" from total samples
    total_samples = confusion_matrix.iloc[:, :10].values.sum()  # Sum only columns 0-9

    # Compute accuracy
    accuracy = true_positives / total_samples if total_samples > 0 else 0

    print("Accuracy (excluding Others and Resets): %.2f%%" % (accuracy * 100))
    return accuracy

def compute_faulty_reset_rates_from_cm(confusion_matrix):
    """
    faulty_rate = (all misclassifications, including 'Others') / total
    reset_rate  = Reset / total
    """
    total = confusion_matrix.values.sum()
    if total == 0:
        return 0.0, 0.0

    reset_count = confusion_matrix["Reset"].sum() if "Reset" in confusion_matrix.columns else 0

    # correct: diagonal for classes 0–9 (if present)
    correct = 0
    for i in range(10):
        if i in confusion_matrix.columns:
            correct += confusion_matrix.loc[i, i]

    faulty_count = total - correct - reset_count
    faulty_count = max(faulty_count, 0)  # just in case of numerical weirdness

    faulty_rate = faulty_count / total
    reset_rate  = reset_count / total

    return faulty_rate, reset_rate


## Base Line

In [ ]:
# csv_path = data_dir + 'image_cnn_run_unprotected_no_fault.csv'
csv_path = "confusion_matrices/tiny_cnn/data/tiny_cnn_unprotected_no_fault_cw.csv"
faulty_data_unprotected = pd.read_csv(csv_path)
print(csv_path)
print("Confusion Matrix (No Protection):")

confusion_matrix = print_confusion_matrix(faulty_data_unprotected)

# Existing metrics
accuracy = print_accuracy(confusion_matrix)
reset_rate_cm = compute_reset_rate(confusion_matrix)  # your previous function

# NEW: faulty + reset rates from CM
faulty_rate, reset_rate = compute_faulty_reset_rates_from_cm(confusion_matrix)

# Store
# start_times_list.append(time)

# if accuracy > 0:
#     inv_accuracy_list.append(1.0 / accuracy)
# else:
#     inv_accuracy_list.append(np.nan)

# faulty_rate_list.append(faulty_rate)
# reset_rate_list.append(reset_rate)

# Normalized confusion matrix for per-start-time plots
confusion_matrix_normalized = normalize_confusion_matrix(confusion_matrix)

fig_name = (
    f"confusion_matrices/tiny_cnn/plots/"
    f"Image_cnn_Confusion_matrix_unprotected_fault.pdf"
)
# print(f"{time:.2f}")

visualize_confusion_matrix_with_accuracy(
    confusion_matrix_normalized,
    accuracy,
    reset_rate_cm,
    fig_name,
    title=f"Confusion Matrix (No Fault)"
)

In [ ]:
# start_time = [(0.22 + (x/1000)) for x in range(0, 21, 1)] 
# start_time = [(0.228 + (x/10000)) for x in range(0, 21, 1)]
# start_time = [0.01*x for x in range(0, 105, 5)] 
# start_time = [0.0040, 0.0228, 0.0229, 0.0230]
start_time = [0.1693]  
# start_time = [0.1022]
filename = "confusion_matrices/tiny_cnn/data/tiny_cnn_unprotected"
# filename = "confusion_matrices/fast_grnn/data/fast_grnn_unprotected"

start_times_list = []
inv_accuracy_list = []
faulty_rate_list = []
reset_rate_list = []

for time in start_time:
    csv_path = filename + f"_{time:.4f}_start_time_run_3_cw.csv"
    faulty_data_unprotected = pd.read_csv(csv_path)
    print(csv_path)
    print("Confusion Matrix (No Protection):")

    confusion_matrix = print_confusion_matrix(faulty_data_unprotected)

    # Existing metrics
    accuracy = print_accuracy(confusion_matrix)
    reset_rate_cm = compute_reset_rate(confusion_matrix)  # your previous function

    # NEW: faulty + reset rates from CM
    faulty_rate, reset_rate = compute_faulty_reset_rates_from_cm(confusion_matrix)

    # Store
    start_times_list.append(time)

    if accuracy > 0:
        inv_accuracy_list.append(1.0 / accuracy)
    else:
        inv_accuracy_list.append(np.nan)

    faulty_rate_list.append(faulty_rate)
    reset_rate_list.append(reset_rate)

    # Normalized confusion matrix for per-start-time plots
    confusion_matrix_normalized = normalize_confusion_matrix(confusion_matrix)

    fig_name = (
        f"confusion_matrices/tiny_cnn/plots/"
        f"Image_cnn_Confusion_matrix_unprotected_fault_{time:.4f}_start_time_run_cw.pdf"
    )
    print(f"{time:.2f}")

    visualize_confusion_matrix_with_accuracy(
        confusion_matrix_normalized,
        accuracy,
        reset_rate_cm,
        fig_name,
        title=f"Confusion Matrix (Fault at {time:.4f}t)"
    )


## Faulty/Reset Rate Plot

In [ ]:
import numpy as np
import pandas as pd
import ast
import os
import seaborn as sns
import matplotlib.pyplot as plt

def parse_predictions(val):
    # Already a Python list
    if isinstance(val, list):
        return val
    # Missing / empty -> no prediction
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []
    # If it's a plain int (or numpy int), wrap it
    if isinstance(val, (int, np.integer)):
        return [int(val)]
    # Strings like "7", "[]", "[1,2]" -> eval safely
    if isinstance(val, str):
        s = val.strip()
        if s == "" or s == "[]":
            return []
        try:
            obj = ast.literal_eval(s)
        except Exception:
            # Fallback: maybe it's "7"
            try:
                return [int(s)]
            except Exception:
                return []
        # Normalize to list
        if isinstance(obj, list):
            return obj
        if isinstance(obj, (int, np.integer)):
            return [int(obj)]
        if isinstance(obj, tuple):
            return list(obj)
        return []
    # Anything else -> no prediction
    return []

def print_confusion_matrix(data_frame):
    # Define the labels for actual and predicted classes
    # actual_labels = list(range(10))
    # predicted_labels = list(range(10)) + ["Others", "Reset"]
    actual_labels = list(range(2))
    predicted_labels = list(range(2)) + ["Others", "Reset"]

    # Initialize an empty confusion matrix with zeros
    CM_df = pd.DataFrame(0, index=actual_labels, columns=predicted_labels)

    # Populate the confusion matrix by iterating over the data
    for _, row in data_frame.iterrows():
        actual = int(row["Actual"])
        predictions = parse_predictions(row["Prediction"])

        # If predictions list is empty, count as "Reset"
        if not predictions:
            CM_df.at[actual, "Reset"] += 1
        else:
            # Check if each predicted class is within the defined range, else mark as "Others"
            for pred in predictions:
                if pred in CM_df.columns:
                    CM_df.at[actual, pred] += 1
                else:
                    CM_df.at[actual, "Others"] += 1
    return CM_df

def normalize_confusion_matrix(CM_df):
    """
    Normalize the confusion matrix rows to percentages.
    """
    CM_df.drop(["Others", "Reset"], axis=1, inplace=True)
    CM_normalized = CM_df.div(CM_df.sum(axis=1), axis=0) * 100  # Convert to percentages
    CM_normalized = CM_normalized.fillna(0)  # Replace NaN with 0 for rows with no samples
    return CM_normalized

def visualize_confusion_matrix(CM_df):
    plt.figure(figsize=(10, 8))  # Adjust the figure size as needed
    sns.heatmap(CM_df, annot=True, fmt="d", cmap="YlGnBu", cbar=True)
    plt.title("Confusion Matrix")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.show()

def visualize_confusion_matrix_with_accuracy(CM_df, accuracy, save_path, title="Confusion Matrix"):
    """
    Visualize the normalized confusion matrix with accuracy displayed in the title.
    """
    
    plt.figure(figsize=(5, 4))
    sns.heatmap(CM_df, annot=True, fmt=".2f", cmap="YlGnBu", cbar=False, annot_kws={"size": 10})
    plt.title(f"{title}\nAccuracy: {accuracy:.2%}", fontsize=14)
    plt.ylabel("Actual", fontsize=14)
    plt.xlabel("Predicted", fontsize=14)
    # plt.savefig(save_path)
    plt.show()

def print_accuracy(confusion_matrix):
    # Calculate true positives (diagonal elements)
    # true_positives = sum(confusion_matrix.iloc[i, i] for i in range(10))  # Only for classes 0-9
    true_positives = sum(confusion_matrix.iloc[i, i] for i in range(2))  # Only for classes 0-9
    
    # Exclude "Others" and "Reset" from total samples
    # total_samples = confusion_matrix.iloc[:, :10].values.sum()  # Sum only columns 0-9
    total_samples = confusion_matrix.iloc[:, :2].values.sum()  # Sum only columns 0-9

    # Compute accuracy
    accuracy = true_positives / total_samples if total_samples > 0 else 0

    print("Accuracy (excluding Others and Resets):%.2f%%" % (accuracy * 100))
    return accuracy



start_time = [0.0795]
start_time = [0.9886]
# filename = "confusion_matrices/tiny_cnn/data/tiny_cnn_unprotected"
# filename = "confusion_matrices/fast_grnn/data/fast_grnn_unprotected"
filename = "confusion_matrices/logistic_regression/data/logistic_regression_unprotected"
# filename = "confusion_matrices/wake_word/data/wake_word_unprotected"
for time in start_time:
    faulty_data_unprotected = pd.read_csv(filename + f"_{time:.4f}_start_time_run_cw.csv")
    print(filename + f"_{time:.4f}_start_time.csv")
    print(f"Confusion Matrix (No Protection):")
    confusion_matrix = print_confusion_matrix(faulty_data_unprotected)
    confusion_matrix_normalized = normalize_confusion_matrix(confusion_matrix)
    accuracy = print_accuracy(confusion_matrix)
    # fig_name = f"confusion_matrices/fast_grnn/plots/Fast_grnn_Confusion_matrix_unprotected_fault_{time:.4f}_start_time.pdf"
    # fig_name = f"confusion_matrices/tiny_cnn/plots/Image_cnn_Confusion_matrix_unprotected_fault_{time:.4f}_start_time.pdf"
    # fig_name = f"confusion_matrices/logistic_regression/plots/Logistic_regression_Confusion_matrix_unprotected_fault_{time:.2f}_start_time.pdf"
    fig_name = f"confusion_matrices/wake_word/plots/Wake_word_Confusion_matrix_unprotected_fault_{time:.2f}_start_time.pdf"
    # time = 24200/105565
    # print(f"{time:.2f}")  
    visualize_confusion_matrix_with_accuracy(confusion_matrix_normalized, accuracy, fig_name, title=f"Confusion Matrix (Fault at {time:.2f}t)")
    
